# CS781 Phase 3 — seven rhetorical-role submissions, Kaggle T4 ×2

Start from **your saved Phase-2 outputs**, retain all four foundation families, and predict one of the 13 official rhetorical roles for **every original sentence**. This notebook embeds its complete source, runs seven adaptation experiments including LoRMA+ and a smaller distilled student, and saves seven platform-format ZIPs. It does not download external model weights or call an LLM.

**Before Run All:** attach the extracted `phase-2-outputs` dataset, choose **GPU T4 ×2**, and attach the seven official Phase-3 files or enable Internet and set an `HF_TOKEN` Kaggle secret after accepting the official dataset terms. The default Phase-2 root is `/kaggle/input/datasets/hardik4tiwari/phase-2-outputs/capstone_phase2`. It loads that directory directly, without archives or recursive checkpoint discovery.

The current Drive brief, updated October 13, states a Phase-3 deadline of **October 16, 2026, 11:59 PM IST**. Leave time for upload and evaluation. Training generates local ZIPs; you must upload the seven individual ZIPs to the course platform. No score is guaranteed; real development and private-test performance must be measured.

**Course-use note:** the brief restricts LLM-written core architecture/tuning code and requires an LLM declaration. This notebook contains AI-written code. Check its permitted use with the teaching team and do not misrepresent authorship. The final research section links the methods and identifies implementation differences.

In [ ]:
import importlib.util, subprocess, sys
requirements = {'tokenizers':'tokenizers>=0.20,<0.24', 'huggingface_hub':'huggingface_hub>=0.27,<3', 'sklearn':'scikit-learn>=1.4,<2'}
missing = [spec for name,spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import torch
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2,3):
    raise RuntimeError('Use a current Kaggle PyTorch image, 2.3+. Preserve its existing CUDA installation.')
print('PyTorch:',torch.__version__)

## Configuration

`PHASE2_ROOT` is the only input-root edit needed for another Kaggle mount. Read the input tree below. `refit_all=False` removes the all-label warmup and adaptation refits to generate all seven candidates sooner. The printed development scores always come from held-out development data.

Default training-loop allowance with refits is at most `14 × (0.04 + 0.15) = 2.66` hours, plus preprocessing, caches, validation, inference, loading and ZIP creation. This is a cap allowance, not an end-to-end runtime prediction. Actual benchmarks run below. Keep settings/GPU counts fixed for resume; use a new `PROJECT` if changing the experiment design.

In [ ]:
from pathlib import Path
import os, json, csv, zipfile, importlib
from IPython.display import display, FileLink
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'capstone_phase3'
CODE = PROJECT / 'code'
CODE.mkdir(parents=True,exist_ok=True)
PHASE2_ROOT = Path('/kaggle/input/datasets/hardik4tiwari/phase-2-outputs/capstone_phase2')
ALLOW_SELECTION_FALLBACK = True
PARENT_OVERRIDES = {}  # Optional: architecture -> explicit full-size Phase-2 best.pt inside PHASE2_ROOT.
DATA_ROOT = None  # Attached Phase_3_RR directory, or its parent; None downloads the official data.
DATA_REVISION = None  # None resolves a commit once; it is then recorded and reused.
SETTINGS = {
    'gpus':2, 'head_gpus':1,
    'chunk_tokens':128, 'block_sentences':24, 'block_overlap':8,
    'head_dim':192, 'chunk_batch':8, 'encoder_checkpointing':True,
    'batch_blocks':1, 'accumulation':2,
    'head_epochs':20, 'head_stage_hours':0.04,
    'adapt_epochs':6, 'adapt_stage_hours':0.15,
    'student_layers':4, 'refit_all':True,
    'token_bidirectional':False,  # Preserve inherited token masks; the sentence head is bidirectional.
}
RUN_BENCHMARK = True
RUN_GPU_SMOKE = True
MAKE_FULL_WORKING_ARCHIVE = True
if torch.cuda.device_count()!=2: raise RuntimeError('Select Kaggle GPU T4 x2 before running.')
for index in range(2):
    gpu=torch.cuda.get_device_properties(index)
    print(index,gpu.name,round(gpu.total_memory/1024**3,2),'GiB')
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='1'
sys.path.insert(0,str(CODE))

## Expected direct Phase-2 input

```text
capstone_phase2/
    phase1/tokenizer.json
    phase1/foundation_manifest.json
    runs/01_decoder_full/refit/best.pt
    runs/02_encoder_decoder_full/refit/best.pt
    runs/03_local_gqa_lora/refit/best.pt
    runs/04_hybrid_partial/refit/best.pt
```

Known `selection/best.pt` paths can substitute if their refits were not run. Every selected source is reported. Other Phase-2 outputs can remain. This notebook reconstructs the original backbone configuration from checkpoint provenance and merges learned Phase-2 low-rank updates. Old `/kaggle/working` paths embedded in a Phase-2 configuration are not followed. Separate Phase-1 input attachment is unnecessary.

## Embedded implementation

The following cells write the complete model, data, training, prediction and workflow source. The token encoder uses your own legal backbone. Long sentences keep all token chunks; a hierarchical context head combines neighboring sentences, document position and CL/IT domain information. A learned CRF decodes the whole document after overlapping-block emissions are merged. The official evaluator is included unchanged for local development checks.

In [ ]:
MODULE_SOURCE = r'''
"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE / 'legal_models.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','legal_models.py')

In [ ]:
MODULE_SOURCE = r'''
"""Official Phase-3 RR parsing, full-sentence token coverage and fixed dev scoring."""
import hashlib,json,math,zipfile
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer

LABELS=['Fact','Issue','ArgumentPetitioner','ArgumentRespondent','Statute','Dissent','PrecedentReliedUpon',
        'PrecedentNotReliedUpon','PrecedentOverruled','RulingByLowerCourt','RatioOfTheDecision','RulingByPresentCourt','None']
DATA_FILES=['Train/CL_train.json','Train/CL_dev.json','Train/IT_train.json','Train/IT_dev.json','Train/label_vocab.json',
            'Test/CL_test.json','Test/IT_test.json']
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def read_json(path):
    def pairs(rows):
        result={}
        for key,value in rows:
            if key in result:raise ValueError(f'Duplicate JSON key: {key}')
            result[key]=value
        return result
    return json.loads(Path(path).read_text(encoding='utf-8-sig'),object_pairs_hook=pairs)
def atomic_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(path)
def atomic_torch(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    torch.save(data,tmp);tmp.replace(path)
def normalized_id(did):
    if not isinstance(did,str) or not did.strip():raise ValueError('Document IDs must be nonempty strings.')
    return did[:-4] if did.endswith('.txt') else did
def label_vocabulary(data):
    if isinstance(data,dict) and 'label2id' in data:data=data['label2id']
    elif isinstance(data,dict) and 'id2label' in data:data=data['id2label']
    if isinstance(data,list):labels=data;ids={i:l for i,l in enumerate(labels)}
    elif isinstance(data,dict) and all(type(v) is int for v in data.values()):
        if len(set(data.values()))!=len(data):raise ValueError('Repeated label IDs.')
        ids={v:k for k,v in data.items()};labels=[ids[i] for i in sorted(ids)]
    elif isinstance(data,dict) and all(isinstance(v,str) for v in data.values()):
        ids={int(k):v for k,v in data.items()};labels=[ids[i] for i in sorted(ids)]
    else:raise ValueError('Unsupported label_vocab.json format.')
    if len(labels)!=13 or len(set(labels))!=13 or set(labels)!=set(LABELS):raise ValueError('Official 13-label vocabulary mismatch.')
    return labels,ids
def _rr_shape(value,depth=0):
    # Diagnostics describe structure only; do not print judgment text.
    if isinstance(value,dict):
        fields=list(value.items())[:6]
        return {str(k):_rr_shape(v,depth+1) if depth<2 else type(v).__name__ for k,v in fields}
    if isinstance(value,list):return {'type':'list','length':len(value),'first':_rr_shape(value[0],depth+1) if value and depth<2 else None}
    return type(value).__name__
def _rr_alias(row,names):
    found=[name for name in names if name in row]
    if len(found)>1 and any(row[name]!=row[found[0]] for name in found[1:]):
        raise ValueError(f'Conflicting fields: {found}')
    return row[found[0]] if found else None
def _rr_document_rows(data):
    if isinstance(data,dict) and len(data)==1 and next(iter(data)) in ('data','documents','records'):
        return _rr_document_rows(next(iter(data.values())))
    if isinstance(data,list):return [(None,row) for row in data]
    if not isinstance(data,dict):raise ValueError(f'Unsupported document container: {_rr_shape(data)}')
    text=_rr_alias(data,('text','sentences'))
    if text is None:return list(data.items())
    did=_rr_alias(data,('id','ids','doc_id','document_id'))
    gold=_rr_alias(data,('labels','label','roles'))
    if isinstance(did,str):return [(None,{'id':did,'text':text,'labels':gold})]
    # Hugging Face Dataset.to_dict(): column name -> one value per document.
    if isinstance(text,list) and isinstance(did,list):
        if len(did)!=len(text) or (gold is not None and (not isinstance(gold,list) or len(gold)!=len(text))):
            raise ValueError('Document columns have different lengths.')
        return [(None,{'id':did[i],'text':sentences,'labels':None if gold is None else gold[i]}) for i,sentences in enumerate(text)]
    # ID-keyed text/label columns, or pandas columns keyed by row index with an explicit id column.
    if isinstance(text,dict):
        for name,column in (('id',did),('labels',gold)):
            if column is not None and (not isinstance(column,dict) or set(column)!=set(text)):
                raise ValueError(f'{name} and text column keys differ.')
        return [(key if did is None else None,{'id':key if did is None else did[key],
                    'text':sentences,'labels':None if gold is None else gold[key]}) for key,sentences in text.items()]
    raise ValueError(f'Document has no explicit string ID, or columns are unsupported: {_rr_shape(data)}')
def _rr_sentence_fields(payload,id_labels):
    if isinstance(payload,dict):
        text=_rr_alias(payload,('text','sentences'))
        gold=_rr_alias(payload,('labels','label','roles'))
    elif isinstance(payload,list):text,gold=payload,None
    else:raise ValueError(f'Document payload must be an object or sentence list; got {type(payload).__name__}.')
    known=lambda x:(type(x) is int and x in id_labels) or (isinstance(x,str) and x in LABELS)
    # Packed document: [sentence_strings, sentence_labels]. Do not confuse two sentence pairs with columns.
    if gold is None and isinstance(text,list) and len(text)==2 and all(isinstance(x,list) for x in text):
        if all(isinstance(x,str) for x in text[0]) and all(known(x) for x in text[1]):
            if all(len(x)==2 and isinstance(x[0],str) and known(x[1]) for x in text):
                raise ValueError('Ambiguous two-row payload; use explicit text and labels fields.')
            text,gold=text
    if not isinstance(text,list):raise ValueError('Expected sentence list; raw document strings are not resegmented.')
    if all(isinstance(x,str) for x in text):return text,gold
    # ID -> [{text/sentence: str, label/role: scalar}, ...] or ID -> [[sentence,label], ...].
    sentences=[];embedded=[];has_label=[]
    for index,item in enumerate(text):
        if isinstance(item,dict):
            sentence=_rr_alias(item,('text','sentence'))
            label=_rr_alias(item,('labels','label','role'))
            present=any(name in item for name in ('labels','label','role'))
        elif isinstance(item,list) and len(item)==2 and isinstance(item[0],str):
            sentence,label=item;present=True
        else:raise ValueError(f'Unsupported sentence entry at position {index}: {_rr_shape(item)}')
        if not isinstance(sentence,str):raise ValueError(f'Sentence {index} must be a string.')
        if present and not known(label):raise ValueError(f'Sentence {index} has an unknown or nonscalar role label.')
        sentences.append(sentence);embedded.append(label);has_label.append(present)
    if any(has_label) and not all(has_label):raise ValueError('Only some sentence entries contain labels.')
    if any(has_label):
        if gold is not None and gold!=embedded:raise ValueError('Sentence labels conflict with document labels.')
        gold=embedded
    return sentences,gold
def documents(data,domain,split,id_labels):
    result={};normalized=set()
    try:
        for key,row in _rr_document_rows(data):
            did=_rr_alias(row,('id','doc_id','document_id')) if isinstance(row,dict) else None
            did=key if did is None else did;nid=normalized_id(did)
            if key is not None and normalized_id(key)!=nid:raise ValueError('Mapping ID differs from document id.')
            if nid in normalized:raise ValueError('Duplicate IDs after official .txt normalization.')
            normalized.add(nid);text,gold=_rr_sentence_fields(row,id_labels)
            if split!='test':
                if not isinstance(gold,list) or len(gold)!=len(text):raise ValueError(f'{did}: label/sentence count mismatch.')
                gold=[id_labels[x] if type(x) is int and x in id_labels else x for x in gold]
                if any(not isinstance(x,str) or x not in LABELS for x in gold):raise ValueError(f'{did}: unknown rhetorical-role label.')
            elif gold is not None:raise ValueError('Public test records must not contain labels.')
            result[did]={'text':text,'gold':gold,'domain':domain,'split':split}
    except ValueError as error:
        raise ValueError(f'{domain}/{split}: {error} JSON structure: {_rr_shape(data)}') from error
    return result
def load_data(root):
    root=Path(root);labels,id_labels=label_vocabulary(read_json(root/'Train/label_vocab.json'))
    docs={};all_ids=set()
    for domain in ('CL','IT'):
        for split in ('train','dev','test'):
            folder='Test' if split=='test' else 'Train'
            local=documents(read_json(root/f'{folder}/{domain}_{split}.json'),domain,split,id_labels)
            norm={normalized_id(did) for did in local}
            if norm&all_ids:raise ValueError('Official normalized IDs overlap across domains/splits.')
            docs.update(local);all_ids|=norm
    return docs,labels
def sentence_blocks(docs,ids,size=12,overlap=4):
    if not 0<=overlap<size:raise ValueError('Invalid sentence-context overlap.')
    rows=[]
    for did in ids:
        n=len(docs[did]['text']);a=0
        while a<n:
            b=min(a+size,n);rows.append((did,a,b))
            if b==n:break
            a=b-overlap
    return rows
def prepare_cache(root,tokenizer_path,path,chunk_tokens=128,block_sentences=12,block_overlap=4):
    docs,labels=load_data(root);tok=Tokenizer.from_file(str(tokenizer_path));tok.no_padding();tok.no_truncation()
    identity=digest({'data':docs,'labels':labels,'tokenizer':file_hash(tokenizer_path),'chunk_tokens':chunk_tokens,
        'block_sentences':block_sentences,'block_overlap':block_overlap,'version':1})
    path=Path(path)
    if path.exists():
        old=torch.load(path,map_location='cpu',weights_only=False)
        if old['fingerprint']!=identity:raise ValueError('Dataset/cache changed; use a fresh PROJECT directory.')
        return old
    dev_hashes={digest(d['text']) for d in docs.values() if d['split']=='dev'}
    removed=[did for did,d in docs.items() if d['split']=='train' and digest(d['text']) in dev_hashes]
    splits={s:[did for did,d in docs.items() if d['split']==s and did not in removed] for s in ('train','dev','test')}
    splits['all']=splits['train']+splits['dev']
    if not splits['train'] or not splits['dev'] or not splits['test']:raise ValueError('Need nonempty public train/dev/test splits.')
    unk=tok.token_to_id('<unk>');unk=0 if unk is None else unk
    audit={'sentences':0,'tokens':0,'chunks':0,'long_sentences':0,'max_sentence_tokens':0,'empty_sentences':0,
           'training_duplicate_documents_removed':removed}
    for d in docs.values():
        d['chunks']=[];d['features']=[];n=len(d['text'])
        for index,sentence in enumerate(d['text']):
            tokens=tok.encode(sentence,add_special_tokens=False).ids
            audit['sentences']+=1;audit['tokens']+=len(tokens);audit['empty_sentences']+=int(not tokens)
            audit['max_sentence_tokens']=max(audit['max_sentence_tokens'],len(tokens));audit['long_sentences']+=int(len(tokens)>chunk_tokens)
            chunks=[tokens[i:i+chunk_tokens] for i in range(0,len(tokens),chunk_tokens)] or [[unk]]
            d['chunks'].append(chunks);audit['chunks']+=len(chunks)
            p=index/max(n-1,1)
            d['features'].append([p,1-p,p*p,float(index==0),float(index==n-1),min(math.log1p(len(tokens))/8,2.)])
        d['target']=None if d['gold'] is None else [labels.index(l) for l in d['gold']]
    cache={'fingerprint':identity,'docs':docs,'labels':labels,'splits':splits,'chunk_tokens':chunk_tokens,
        'tokenizer_sha256':file_hash(tokenizer_path),'block_sentences':block_sentences,'block_overlap':block_overlap,'audit':audit}
    for split,ids in splits.items():cache[split+'_blocks']=sentence_blocks(docs,ids,block_sentences,block_overlap)
    cache['support']={split:{l:sum(docs[did]['gold'].count(l) for did in splits[split]) for l in labels} for split in ('train','dev')}
    atomic_torch(path,cache);return cache

class Blocks(Dataset):
    def __init__(self,cache,split,features=None,teacher=None):
        self.cache,self.rows,self.features,self.teacher=cache,cache[split+'_blocks'],features,teacher
    def __len__(self):return len(self.rows)
    def __getitem__(self,index):
        did,a,b=self.rows[index];doc=self.cache['docs'][did]
        return {'did':did,'start':a,'end':b,'chunks':doc['chunks'][a:b],
            'position':doc['features'][a:b],'domain':0 if doc['domain']=='CL' else 1,
            'target':doc['target'][a:b] if doc['target'] is not None else None,
            'features':self.features[did][a:b].float() if self.features is not None else None,
            'teacher':self.teacher[did][a:b].float() if self.teacher is not None else None}
def collate(rows):
    B=len(rows);S=max(x['end']-x['start'] for x in rows)
    positions=torch.zeros(B,S,6);mask=torch.zeros(B,S,dtype=torch.bool);targets=torch.full((B,S),-100,dtype=torch.long)
    chunks=[];owners=[];lengths=[];features=None;teacher=None
    if rows[0]['features'] is not None:features=torch.zeros(B,S,rows[0]['features'].size(-1))
    if rows[0]['teacher'] is not None:teacher=torch.zeros(B,S,rows[0]['teacher'].size(-1))
    for row,item in enumerate(rows):
        n=item['end']-item['start'];mask[row,:n]=True;positions[row,:n]=torch.tensor(item['position'])
        if item['target'] is not None:targets[row,:n]=torch.tensor(item['target'])
        if features is not None:features[row,:n]=item['features']
        else:
            for sentence,parts in enumerate(item['chunks']):
                for part in parts:chunks.append(part);owners.append(row*S+sentence);lengths.append(len(part))
        if teacher is not None:teacher[row,:n]=item['teacher']
    T=max(map(len,chunks),default=1);ids=torch.zeros(len(chunks),T,dtype=torch.long);valid=torch.zeros_like(ids,dtype=torch.bool)
    for i,chunk in enumerate(chunks):ids[i,:len(chunk)]=torch.tensor(chunk);valid[i,:len(chunk)]=True
    return {'ids':ids,'valid':valid,'owners':torch.tensor(owners,dtype=torch.long),'chunk_lengths':torch.tensor(lengths,dtype=torch.float32),
        'mask':mask,'position':positions,'domain':torch.tensor([r['domain'] for r in rows]),'targets':targets,
        'features':features,'teacher':teacher,'rows':rows}
def to_device(batch,device):return {k:v.to(device,non_blocking=True) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
def metric(prediction,truth,labels):
    if set(prediction)!=set(truth):raise ValueError('Metric document IDs differ.')
    matrix=np.zeros((len(labels),len(labels)),dtype=np.int64);ids={l:i for i,l in enumerate(labels)}
    for did,gold in truth.items():
        pred=prediction[did]
        if len(pred)!=len(gold):raise ValueError('Sentence-count mismatch.')
        for g,p in zip(gold,pred):matrix[ids[g],ids[p]]+=1
    tp=np.diag(matrix);support=matrix.sum(1);predicted=matrix.sum(0)
    precision=tp/np.maximum(predicted,1);recall=tp/np.maximum(support,1);f1=2*tp/np.maximum(support+predicted,1)
    return {'macro_f1':float(f1.mean()),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),
        'per_label':[{'label':l,'support':int(support[i]),'predicted':int(predicted[i]),'f1':float(f1[i]),
         'precision':float(precision[i]),'recall':float(recall[i])} for i,l in enumerate(labels)],'confusion_matrix':matrix.tolist()}
def validate_prediction(pred,docs,labels):
    if set(pred)!=set(docs):raise ValueError('Predictions must include every official ID exactly once.')
    norm=[normalized_id(did) for did in pred]
    if len(set(norm))!=len(norm):raise ValueError('IDs collide after official normalization.')
    for did,values in pred.items():
        if not isinstance(values,list) or len(values)!=len(docs[did]['text']):raise ValueError(f'{did}: sentence-count mismatch.')
        if any(x not in labels for x in values):raise ValueError(f'{did}: invalid label.')
def submission_zip(pred,docs,labels,path):
    validate_prediction(pred,docs,labels);path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    data=json.dumps(pred,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()
    tmp=path.with_suffix('.tmp.zip')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:z.writestr('prediction.json',data)
    tmp.replace(path)
    with zipfile.ZipFile(path) as z:
        if z.namelist()!=['prediction.json']:raise ValueError('Incorrect ZIP root.')
        validate_prediction(json.loads(z.read('prediction.json')),docs,labels)
    return digest(pred)
'''
(CODE / 'rr_data.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','rr_data.py')

In [ ]:
MODULE_SOURCE = r'''
"""Own Phase-2 backbones, hierarchical RR heads, learned CRF and five adaptation routes."""
import math,types
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from legal_models import LegalLM,ModelConfig,Attention

class LowRank(nn.Module):
    def __init__(self,base,rank,alpha,kind):
        super().__init__();self.base=base;self.kind=kind
        self.scale=alpha/(math.sqrt(rank) if kind=='rslora' else rank)
        self.A=nn.Linear(base.out_features if kind=='lorma' else base.in_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight);base.requires_grad_(False)
    def forward(self,x):
        y=self.base(x);return y+self.scale*self.B(self.A(y if self.kind=='lorma' else x))
    def merged(self):
        delta=self.B.weight@(self.A.weight@self.base.weight) if self.kind=='lorma' else self.B.weight@self.A.weight
        return self.base.weight+self.scale*delta
class IA3(nn.Module):
    def __init__(self,base,on_input=False):
        super().__init__();self.base=base;self.on_input=on_input;base.requires_grad_(False)
        self.scale=nn.Parameter(torch.ones(base.in_features if on_input else base.out_features))
    def forward(self,x):return self.base(x*self.scale) if self.on_input else self.base(x)*self.scale

def attention_forward(self,x,memory=None,past=None,position=0,use_cache=False):
    if past is not None or use_cache:raise ValueError('RR is full-sequence classification.')
    b,t,_=x.shape;source=x if memory is None else memory;s=source.size(1)
    q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)
    k=self.k(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    v=self.v(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    if self.rotary is not None:q=self.rotary(q);k=self.rotary(k)
    if self.kv_heads!=self.heads:
        k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    valid=self._rr_valid;causal=self.causal and not self._rr_bidirectional
    window=self.window if causal else 0;mask=None
    if window or not bool(valid.all()):
        qi=torch.arange(t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]
        allowed=ki<=qi if causal else torch.ones(t,s,dtype=torch.bool,device=x.device)
        if window:allowed=allowed&((qi-ki)<window)
        mask=allowed[None,None]&valid[:,None,None,:s];causal=False
    output=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,dropout_p=self.dropout if self.training else 0.)
    return self.out(output.transpose(1,2).contiguous().view(b,t,-1)),None

class CRF(nn.Module):
    def __init__(self,count):
        super().__init__();self.start=nn.Parameter(torch.zeros(count));self.end=nn.Parameter(torch.zeros(count))
        self.transitions=nn.Parameter(torch.zeros(count,count))
    def nll(self,emissions,target,mask):
        e=emissions.float();tags=target.clamp_min(0);B,S,C=e.shape;row=torch.arange(B,device=e.device)
        score=self.start[tags[:,0]]+e[row,0,tags[:,0]];forward=self.start+e[:,0]
        for i in range(1,S):
            step=self.transitions[tags[:,i-1],tags[:,i]]+e[row,i,tags[:,i]]
            score=score+step*mask[:,i]
            update=torch.logsumexp(forward[:,:,None]+self.transitions[None]+e[:,i,None,:],dim=1)
            forward=torch.where(mask[:,i,None],update,forward)
        last=tags.gather(1,(mask.sum(1)-1)[:,None]).squeeze(1)
        score=score+self.end[last];partition=torch.logsumexp(forward+self.end,1)
        return ((partition-score)/mask.sum(1)).mean()
    def arrays(self):return [self.start.detach().float().cpu().numpy(),self.transitions.detach().float().cpu().numpy(),self.end.detach().float().cpu().numpy()]
    @staticmethod
    def decode(emissions,arrays,transition_scale=1.,bias=None):
        n,c=emissions.shape
        if not n:return []
        start,transitions,end=arrays;values=emissions.astype(np.float32)
        if bias is not None:values=values+np.asarray(bias,dtype=np.float32)
        score=values[0]+start*transition_scale;back=[]
        for row in values[1:]:
            options=score[:,None]+transitions*transition_scale;previous=options.argmax(0)
            score=options[previous,np.arange(c)]+row;back.append(previous)
        tag=int((score+end*transition_scale).argmax());tags=[tag]
        for previous in reversed(back):tag=int(previous[tag]);tags.append(tag)
        return tags[::-1]

class RRModel(nn.Module):
    def __init__(self,backbone,cfg,label_count):
        super().__init__();self.backbone=backbone;self.cfg=cfg;d=backbone.config.dim
        method=cfg['method'];backbone.requires_grad_(method in ('full','kd'))
        if method in ('lora','rslora','lorma'):
            targets={'q','v','out','gate','up','down'}
            replacements=[(name,module) for name,module in backbone.named_modules() if isinstance(module,nn.Linear) and name.rsplit('.',1)[-1] in targets]
            for name,module in replacements:
                parent,attribute=name.rsplit('.',1)
                setattr(backbone.get_submodule(parent),attribute,LowRank(module,cfg.get('rank',8),cfg.get('alpha',16),method))
        elif method=='ia3':
            replacements=[(name,module) for name,module in backbone.named_modules() if isinstance(module,nn.Linear) and name.rsplit('.',1)[-1] in ('k','v','down')]
            for name,module in replacements:
                parent,attribute=name.rsplit('.',1);setattr(backbone.get_submodule(parent),attribute,IA3(module,attribute=='down'))
        elif method not in ('full','kd'):raise ValueError(method)
        self.attention_modules=[m for m in backbone.modules() if isinstance(m,Attention)]
        for module in self.attention_modules:
            module.forward=types.MethodType(attention_forward,module);module.dropout=cfg.get('encoder_dropout',.05)
            module._rr_bidirectional=cfg.get('token_bidirectional',False)
        for module in backbone.modules():
            if isinstance(module,nn.Dropout):module.p=cfg.get('encoder_dropout',.05)
        backbone.config.gradient_checkpointing=False
        if cfg.get('freeze_backbone'):backbone.requires_grad_(False)
        h=cfg.get('head_dim',192);self.pool_dim=3*d
        self.domain=nn.Embedding(2,8);self.projection=nn.Sequential(nn.LayerNorm(3*d+14),nn.Linear(3*d+14,h),nn.GELU(),nn.Dropout(cfg.get('dropout',.25)))
        self.head_type=cfg.get('head_type','bilstm')
        if self.head_type=='transformer':
            layer=nn.TransformerEncoderLayer(h,4,4*h,dropout=cfg.get('dropout',.25),activation='gelu',batch_first=True,norm_first=True)
            self.context=nn.TransformerEncoder(layer,2,enable_nested_tensor=False);width=h
        else:
            self.context=nn.LSTM(h,h//2,num_layers=2,batch_first=True,bidirectional=True,dropout=cfg.get('dropout',.25));width=h
        self.output_norm=nn.LayerNorm(width);self.drop=nn.Dropout(cfg.get('dropout',.25))
        self.classifier=nn.Linear(width,label_count);self.shift=nn.Linear(width*3,1);self.crf=CRF(label_count)
    def pool_chunk(self,ids,valid):
        for module in self.attention_modules:module._rr_valid=valid
        hidden=self.backbone(ids,source_ids=ids) if self.backbone.config.architecture=='encoder_decoder' else self.backbone(ids)
        lengths=valid.sum(1);mask=valid[:,:,None]
        mean=(hidden.float()*mask).sum(1)/lengths[:,None]
        maximum=hidden.float().masked_fill(~mask,-1e9).max(1).values
        last=hidden[torch.arange(len(ids),device=ids.device),lengths-1].float()
        return torch.cat((mean,maximum,last),-1)
    def encode(self,batch):
        if batch['features'] is not None:return batch['features']
        pieces=[];step=self.cfg.get('chunk_batch',8)
        for first in range(0,len(batch['ids']),step):
            ids=batch['ids'][first:first+step];valid=batch['valid'][first:first+step]
            if self.training and self.cfg.get('encoder_checkpointing',True) and any(p.requires_grad for p in self.backbone.parameters()):
                value=checkpoint(self.pool_chunk,ids,valid,use_reentrant=False)
            else:value=self.pool_chunk(ids,valid)
            pieces.append(value)
        vectors=torch.cat(pieces);B,S=batch['mask'].shape
        pooled=vectors.new_zeros(B*S,self.pool_dim);counts=vectors.new_zeros(B*S,1)
        weights=batch['chunk_lengths'][:,None].to(vectors.dtype)
        pooled=pooled.index_add(0,batch['owners'],vectors*weights);counts=counts.index_add(0,batch['owners'],weights)
        return (pooled/counts.clamp_min(1)).reshape(B,S,-1)
    def forward(self,batch):
        pooled=self.encode(batch);B,S,_=pooled.shape
        domain=self.domain(batch['domain'])[:,None].expand(-1,S,-1)
        hidden=self.projection(torch.cat((pooled,batch['position'],domain),-1))
        if self.head_type=='transformer':hidden=self.context(hidden,src_key_padding_mask=~batch['mask'])
        else:
            packed=nn.utils.rnn.pack_padded_sequence(hidden,batch['mask'].sum(1).cpu(),batch_first=True,enforce_sorted=False)
            hidden,_=self.context(packed);hidden,_=nn.utils.rnn.pad_packed_sequence(hidden,batch_first=True,total_length=S)
        hidden=self.drop(self.output_norm(hidden));emissions=self.classifier(hidden)
        pairs=torch.cat((hidden[:,1:],hidden[:,:-1],(hidden[:,1:]-hidden[:,:-1]).abs()),-1)
        return {'logits':emissions,'shift':self.shift(pairs).squeeze(-1)}

def build_model(cfg,restore=None):
    parent=torch.load(cfg['parent'],map_location='cpu',weights_only=False);config=ModelConfig(**parent['config'])
    state=parent['model'];retained=None
    if cfg['method']=='kd':
        count=cfg.get('student_layers',4);original=config.layers
        if config.architecture!='decoder' or not 1<=count<original:raise ValueError('Student must retain fewer original decoder blocks.')
        retained=[round(i*(original-1)/(count-1)) for i in range(count)] if count>1 else [original-1]
        state={key:value for key,value in state.items() if not key.startswith('blocks.')}
        for new,old in enumerate(retained):
            prefix=f'blocks.{old}.'
            state.update({f'blocks.{new}.'+k[len(prefix):]:v for k,v in parent['model'].items() if k.startswith(prefix)})
        config.layers=count
    backbone=LegalLM(config);backbone.load_state_dict(state,strict=True);model=RRModel(backbone,cfg,cfg['label_count'])
    if restore:
        saved=torch.load(restore,map_location='cpu',weights_only=False)
        missing,unexpected=model.load_state_dict(saved['model'],strict=not saved.get('delta_only',False))
        trainable={name for name,p in model.named_parameters() if p.requires_grad}
        if unexpected or (set(missing)&trainable):raise ValueError('Incomplete RR checkpoint.')
    elif cfg.get('head_init'):
        trained=torch.load(cfg['head_init'],map_location='cpu',weights_only=False)['model']
        head={name:value for name,value in trained.items() if not name.startswith('backbone.')}
        missing,unexpected=model.load_state_dict(head,strict=False)
        if unexpected or any(not name.startswith('backbone.') for name in missing):raise ValueError('Head warmup checkpoint is incompatible.')
    return model,{**parent['provenance'],'retained_parent_blocks':retained,'backbone_config':vars(config)}
def parameter_report(model):return {'total':sum(p.numel() for p in model.parameters()),'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
    'backbone':sum(p.numel() for p in model.backbone.parameters())}
def loss_function(model,result,batch,weights,cfg):
    e=result['logits'].float();targets=batch['targets'];mask=batch['mask']
    crf=model.crf.nll(e,targets,mask)
    ce=F.cross_entropy(e.flatten(0,1),targets.flatten(),weight=weights,ignore_index=-100,label_smoothing=.03)
    shift_mask=mask[:,1:]&mask[:,:-1]
    if bool(shift_mask.any()):
        y=(targets[:,1:]!=targets[:,:-1]).float()
        shift=F.binary_cross_entropy_with_logits(result['shift'].float()[shift_mask],y[shift_mask])
    else:shift=result['shift'].float().sum()*0
    supervised=crf+cfg.get('ce_weight',.5)*ce+cfg.get('shift_weight',.15)*shift
    if batch['teacher'] is not None:
        T=cfg.get('temperature',2.);student=e[mask];teacher=batch['teacher'][mask]
        kd=F.kl_div(F.log_softmax(student/T,-1),F.softmax(teacher.float()/T,-1),reduction='batchmean')*T*T
        return (1-cfg.get('kd_alpha',.5))*supervised+cfg.get('kd_alpha',.5)*kd
    return supervised
'''
(CODE / 'rr_models.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','rr_models.py')

In [ ]:
MODULE_SOURCE = r'''
"""Distributed feature/logit caches; full-document CRF decoding without sentence omissions."""
import argparse,contextlib,json,os
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from rr_data import Blocks,collate,to_device,atomic_torch,atomic_json,digest,file_hash,metric,read_json
from rr_models import build_model,CRF
def setup():
    rank=int(os.environ.get('RANK',0));local=int(os.environ.get('LOCAL_RANK',0));world=int(os.environ.get('WORLD_SIZE',1))
    device=torch.device('cuda',local) if torch.cuda.is_available() else torch.device('cpu')
    if device.type=='cuda':torch.cuda.set_device(local)
    if world>1 and not dist.is_initialized():
        if device.type=='cuda':dist.init_process_group('nccl',device_id=device)
        else:dist.init_process_group('gloo')
    return rank,world,device
def autocast(device):return torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()
@torch.inference_mode()
def sentence_features(model,cache,did,device):
    doc=cache['docs'][did];n=len(doc['text'])
    if not n:return torch.zeros(0,model.pool_dim)
    local=dict(cache);local['infer_blocks']=[(did,0,n)];dataset=Blocks(local,'infer')
    batch=to_device(collate([dataset[0]]),device)
    with autocast(device):values=model.encode(batch)[0]
    return values.float().cpu()
@torch.inference_mode()
def logits_for_documents(model,cache,ids,device,features=None):
    model.eval();output={}
    for did in ids:
        n=len(cache['docs'][did]['text'])
        if not n:output[did]=torch.zeros(0,len(cache['labels']));continue
        vector=features[did].float() if features is not None else sentence_features(model,cache,did,device)
        local=dict(cache);local['infer_blocks']=[row for row in cache['all_blocks']+cache['test_blocks'] if row[0]==did]
        dataset=Blocks(local,'infer',features={did:vector});emissions=np.zeros((n,len(cache['labels'])),np.float32);counts=np.zeros(n,np.float32)
        for first in range(0,len(dataset),model.cfg.get('head_batch',8)):
            batch=to_device(collate([dataset[i] for i in range(first,min(first+model.cfg.get('head_batch',8),len(dataset)))]),device)
            with autocast(device):scores=model(batch)['logits'].float().cpu().numpy()
            for j,row in enumerate(batch['rows']):
                a,b=row['start'],row['end'];length=b-a
                weight=np.minimum(np.arange(1,length+1),np.arange(length,0,-1)).astype(np.float32)
                weight=np.minimum(weight,cache['block_overlap']+1)
                emissions[a:b]+=scores[j,:length]*weight[:,None];counts[a:b]+=weight
        if np.any(counts<=0):raise ValueError('Inference omitted sentences.')
        output[did]=torch.from_numpy(emissions/counts[:,None])
    return output
def decode(logits,arrays,labels,calibration=None):
    cal=calibration or {'transition_scale':1.,'bias':[0.]*len(labels)}
    return {did:[labels[i] for i in CRF.decode(values.float().numpy() if isinstance(values,torch.Tensor) else values,
        arrays,cal['transition_scale'],cal['bias'])] for did,values in logits.items()}
def select_calibration(logits,arrays,cache):
    labels=cache['labels'];truth={did:cache['docs'][did]['gold'] for did in cache['splits']['dev']}
    counts=np.array([cache['support']['train'][label]+1 for label in labels],np.float32)
    log_prior=np.log(counts/counts.sum());best=None
    for scale in (1.,.5,0.):
        for amount in (0.,.15,.30):
            bias=(-amount*(log_prior-log_prior.mean())).tolist();cal={'transition_scale':scale,'bias':bias}
            pred=decode(logits,arrays,labels,cal);score=metric(pred,truth,labels)
            if best is None or score['macro_f1']>best[0]['macro_f1']+1e-9:best=(score,cal,pred)
    return best
def domain_metrics(pred,cache):
    output={}
    for domain in ('CL','IT'):
        truth={did:cache['docs'][did]['gold'] for did in pred if cache['docs'][did]['domain']==domain}
        output[domain]=metric({did:pred[did] for did in truth},truth,cache['labels'])
    return output
def main():
    p=argparse.ArgumentParser();p.add_argument('--action',choices=['features','logits'],required=True)
    p.add_argument('--config');p.add_argument('--checkpoint');p.add_argument('--cache',required=True)
    p.add_argument('--split',choices=['train','dev','all','test','every'],default='every');p.add_argument('--output',required=True)
    args=p.parse_args();rank,world,device=setup();cache=torch.load(args.cache,map_location='cpu',weights_only=False)
    saved=torch.load(args.checkpoint,map_location='cpu',weights_only=False) if args.checkpoint else None
    cfg=saved['config'] if saved else json.loads(Path(args.config).read_text());model,_=build_model(cfg,args.checkpoint);model.to(device).eval()
    if saved and saved['cache_fingerprint']!=cache['fingerprint']:raise ValueError('Logit cache data mismatch.')
    ids=list(cache['docs']) if args.split=='every' else cache['splits'][args.split]
    origin=file_hash(args.checkpoint) if args.checkpoint else digest([file_hash(cfg['parent']),cfg.get('student_layers') if cfg['method']=='kd' else None,cfg.get('token_bidirectional',False)])
    identity=digest([args.action,origin,cache['fingerprint'],ids,world])
    root=Path(args.output);root.mkdir(parents=True,exist_ok=True);path=root/f'part_{rank}.pt'
    marker=root/'complete.json'
    if marker.exists() and (root/'complete.pt').exists():
        if read_json(marker)['identity']!=identity:raise ValueError('Completed feature/logit cache identity changed.')
        if rank==0:print('Already cached:',root,flush=True)
        if world>1:dist.destroy_process_group()
        return
    previous=torch.load(path,map_location='cpu',weights_only=False) if path.exists() else {'identity':identity,'values':{}}
    if previous['identity']!=identity:raise ValueError('Feature/logit cache identity changed. Use a new PROJECT directory.')
    values=previous['values']
    pending=0
    for did in ids[rank::world]:
        if did in values:continue
        value=sentence_features(model,cache,did,device) if args.action=='features' else logits_for_documents(model,cache,[did],device)[did]
        if not bool(torch.isfinite(value).all()):raise FloatingPointError('Non-finite cached representation.')
        values[did]=value.half();pending+=1
        if pending>=8:atomic_torch(path,{'identity':identity,'values':values});pending=0
    atomic_torch(path,{'identity':identity,'values':values})
    if world>1:dist.barrier()
    if rank==0:
        merged={}
        for i in range(world):merged.update(torch.load(root/f'part_{i}.pt',map_location='cpu',weights_only=False)['values'])
        if set(merged)!=set(ids):raise ValueError('Cache is incomplete.')
        payload={'identity':identity,'values':merged,'cache_fingerprint':cache['fingerprint'],'source_sha256':origin,
            'split':args.split,'kind':args.action,'crf':model.crf.arrays() if args.action=='logits' else None,'complete':True}
        atomic_torch(root/'complete.pt',payload)
        atomic_json(marker,{'identity':identity,'complete':True,'documents':len(ids)})
        for i in range(world):(root/f'part_{i}.pt').unlink(missing_ok=True)
    if world>1:dist.barrier();dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE / 'rr_predict.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','rr_predict.py')

In [ ]:
MODULE_SOURCE = r'''
"""Resumable two-GPU head warmup and short full/PEFT/KD RR adaptation."""
import argparse,contextlib,json,math,random,time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader,DistributedSampler
from rr_data import Blocks,collate,to_device,digest,atomic_json,atomic_torch,metric,file_hash
from rr_models import build_model,loss_function,parameter_report
from rr_predict import setup,autocast,logits_for_documents,decode
def rng_state(device):return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
    'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}
def restore_rng(state,device):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if device.type=='cuda':torch.cuda.set_rng_state(state['cuda'],device)
def main():
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);p.add_argument('--cache',required=True)
    p.add_argument('--output',required=True);p.add_argument('--split',choices=['train','all'],default='train')
    p.add_argument('--features');p.add_argument('--teacher');p.add_argument('--epochs',type=int);p.add_argument('--smoke',action='store_true')
    args=p.parse_args();rank,world,device=setup();cfg=json.loads(Path(args.config).read_text());root=Path(args.output);root.mkdir(parents=True,exist_ok=True)
    seed=cfg.get('seed',230437);random.seed(seed+rank);np.random.seed(seed+rank);torch.manual_seed(seed+rank)
    cache=torch.load(args.cache,map_location='cpu',weights_only=False)
    if bool(args.features)!=bool(cfg.get('freeze_backbone',False)):
        raise ValueError('Frozen feature caches are used only for frozen-backbone warmup.')
    def load_aux(path,kind):
        if not path:return None
        value=torch.load(path,map_location='cpu',weights_only=False)
        if not value['complete'] or value['cache_fingerprint']!=cache['fingerprint'] or value['kind']!=kind:raise ValueError('Auxiliary cache mismatch.')
        if kind=='features':
            expected=digest([file_hash(cfg['parent']),cfg.get('student_layers') if cfg['method']=='kd' else None,cfg.get('token_bidirectional',False)])
            if value['source_sha256']!=expected:raise ValueError('Frozen features came from another encoder or attention configuration.')
        if kind=='logits' and value['split']!=args.split:raise ValueError('KD teacher must match the current training split.')
        if any(did not in value['values'] for did in cache['splits'][args.split]):raise ValueError('Auxiliary cache omits training documents.')
        return value['values']
    features=load_aux(args.features,'features');teacher=load_aux(args.teacher,'logits')
    if cfg['method']=='kd' and teacher is None and not args.smoke:raise ValueError('The student requires a larger RR teacher logit cache.')
    dataset=Blocks(cache,args.split,features,teacher)
    if not len(dataset):raise ValueError('Empty training blocks.')
    sampler=DistributedSampler(dataset,world,rank,shuffle=True,seed=seed,drop_last=False);generator=torch.Generator()
    loader=DataLoader(dataset,batch_size=cfg.get('batch_blocks',1),sampler=sampler,collate_fn=collate,
        num_workers=0,pin_memory=device.type=='cuda',generator=generator)
    model,provenance=build_model(cfg);model.to(device);report=parameter_report(model)
    if cfg['method']=='kd' and cfg.get('teacher_parameters',float('inf'))<=report['total']:raise ValueError('RR teacher must be larger than the student.')
    backbone=[];head=[]
    for name,param in model.named_parameters():
        if param.requires_grad:(backbone if name.startswith('backbone.') else head).append(param)
    optimizer=torch.optim.AdamW([{'params':backbone,'lr':cfg['encoder_lr'],'base_lr':cfg['encoder_lr']},
        {'params':head,'lr':cfg['head_lr'],'base_lr':cfg['head_lr']}],weight_decay=cfg.get('weight_decay',.03),fused=device.type=='cuda')
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
    support=np.array([sum(cache['docs'][did]['gold'].count(label) for did in cache['splits'][args.split])+1 for label in cache['labels']],dtype=np.float32)
    weights=np.sqrt(support.sum()/support);weights=np.clip(weights/weights.mean(),.25,3.)
    weights=torch.tensor(weights,device=device)
    epochs=args.epochs or cfg['epochs'];accum=cfg.get('accumulation',2)
    planned=math.ceil(len(loader)/accum)*cfg.get('schedule_epochs',cfg['epochs'])
    identity=digest({'cfg':cfg,'cache':cache['fingerprint'],'split':args.split,'epochs':epochs,'world':world,
        'features':file_hash(args.features) if args.features else None,'teacher':file_hash(args.teacher) if args.teacher else None,
        'parent':file_hash(cfg['parent']),'head_init':file_hash(cfg['head_init']) if cfg.get('head_init') else None,'smoke':args.smoke})
    latest=root/'latest.pt';summary_path=root/'summary.json';best=root/'best.pt'
    if summary_path.exists():
        summary=json.loads(summary_path.read_text())
        if summary['identity']!=identity:raise ValueError('Run configuration changed; use a fresh PROJECT directory.')
        if summary.get('complete') and best.exists():
            if rank==0:print('Already complete:',root,flush=True)
            if world>1:dist.destroy_process_group()
            return
    epoch=cursor=updates=skipped=bad=0;best_f1=-1.;best_epoch=0;history=[];elapsed_prior=0.
    if latest.exists():
        state=torch.load(latest,map_location='cpu',weights_only=False)
        if state['identity']!=identity:raise ValueError('Resume identity changed.')
        model.load_state_dict(state['model'],strict=not state['delta_only']);optimizer.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler'])
        epoch=state['epoch'];cursor=state['cursor'];updates=state['updates'];skipped=state['skipped'];bad=state['bad']
        best_f1=state['best_f1'];best_epoch=state['best_epoch'];history=state['history'];elapsed_prior=state['elapsed']
        restore_rng(state['rng'][rank],device)
    wrapped=DDP(model,device_ids=[device.index] if device.type=='cuda' else None,broadcast_buffers=False) if world>1 else model
    started=time.monotonic();stopped=False
    trainable={name for name,param in model.named_parameters() if param.requires_grad}
    delta_only=cfg.get('freeze_backbone',False) or cfg['method'] not in ('full','kd')
    def save(is_best=False,complete=False):
        states=[None]*world
        if world>1:dist.all_gather_object(states,rng_state(device))
        else:states[0]=rng_state(device)
        if rank==0:
            weights_state=model.state_dict()
            if delta_only:weights_state={key:value for key,value in weights_state.items() if key in trainable}
            light={'model':weights_state,'delta_only':delta_only,'config':cfg,'provenance':provenance,'parameters':report,
                'cache_fingerprint':cache['fingerprint'],'labels':cache['labels'],'training_origin':'own_phase1_phase2_phase3_rr',
                'stage':'selection' if args.split=='train' else 'all_label_refit','epoch':epoch,'updates':updates}
            if is_best:atomic_torch(best,light)
            state={**light,'identity':identity,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'rng':states,
                'cursor':cursor,'skipped':skipped,'bad':bad,'best_f1':best_f1,'best_epoch':best_epoch,'history':history,
                'elapsed':elapsed_prior+time.monotonic()-started}
            atomic_torch(latest,state)
            atomic_json(summary_path,{'identity':identity,'complete':complete,'parameters':report,'epoch':epoch,'updates':updates,
                'best_epoch':best_epoch,'best_f1':best_f1,'elapsed_seconds':state['elapsed'],'history':history,'amp_skipped_updates':skipped,
                'stop_reason':'time_cap' if stopped else 'epochs_or_patience' if complete else 'running'})
        if world>1:dist.barrier()
    if rank==0:print(json.dumps({'experiment':cfg['name'],'stage':args.split,'frozen_features':bool(features is not None),
        'blocks':len(dataset),'parameters':report,'world_size':world}),flush=True)
    while epoch<epochs:
        sampler.set_epoch(epoch);generator.manual_seed(seed+epoch);iterator=iter(loader)
        for _ in range(cursor):next(iterator)
        model.train();loss_sum=0.;loss_count=0
        while cursor<len(loader):
            micro=[next(iterator) for _ in range(min(accum,len(loader)-cursor))];optimizer.zero_grad(set_to_none=True)
            warmup=max(1,round(planned*.1));step=updates+1
            factor=step/warmup if step<=warmup else .1+.9*.5*(1+math.cos(math.pi*min(1.,(step-warmup)/max(1,planned-warmup))))
            for group in optimizer.param_groups:group['lr']=group['base_lr']*factor
            for i,batch in enumerate(micro):
                batch=to_device(batch,device)
                if args.smoke and cfg['method']=='kd' and batch['teacher'] is None:
                    batch['teacher']=torch.zeros(*batch['mask'].shape,len(cache['labels']),device=device)
                sync=wrapped.no_sync() if world>1 and i+1<len(micro) else contextlib.nullcontext()
                with sync:
                    with autocast(device):
                        result=wrapped(batch);loss=loss_function(model,result,batch,weights,cfg)
                        if not bool(torch.isfinite(loss)):raise FloatingPointError('Non-finite loss; inspect FP16, data and saved state.')
                    scaler.scale(loss/len(micro)).backward()
                loss_sum+=float(loss.detach());loss_count+=1
            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            before=scaler.get_scale();scaler.step(optimizer);scaler.update()
            if scaler.get_scale()<before:skipped+=1
            else:updates+=1
            cursor+=len(micro)
            flag=torch.tensor(int(elapsed_prior+time.monotonic()-started>=cfg.get('max_stage_hours',.15)*3600 or (args.smoke and updates>=1)),device=device)
            if world>1:dist.all_reduce(flag,op=dist.ReduceOp.MAX)
            stopped=bool(flag)
            if updates and updates%cfg.get('checkpoint_every',100)==0:save()
            if rank==0 and updates and updates%cfg.get('log_every',20)==0:
                print(json.dumps({'epoch':epoch+1,'updates':updates,'loss':loss_sum/max(loss_count,1),'gradient_norm':float(norm),
                    'elapsed_hours':(elapsed_prior+time.monotonic()-started)/3600,'amp_skipped_updates':skipped}),flush=True)
            if stopped:break
        score=None
        if world>1:dist.barrier()
        if args.split=='train' and rank==0 and not args.smoke:
            values=logits_for_documents(model,cache,cache['splits']['dev'],device,features)
            pred=decode(values,model.crf.arrays(),cache['labels']);truth={did:cache['docs'][did]['gold'] for did in pred}
            score=metric(pred,truth,cache['labels']);atomic_json(root/'dev_prediction.json',pred)
            print(json.dumps({'epoch':epoch+1,'dev_macro_f1':score['macro_f1']}),flush=True)
        if world>1:
            box=[score];dist.broadcast_object_list(box,src=0);score=box[0]
        improved=args.split=='all' or args.smoke or score['macro_f1']>best_f1+1e-8
        if improved:best_f1=score['macro_f1'] if score else -1.;best_epoch=epoch+1;bad=0
        else:bad+=1
        history.append({'epoch':epoch+1,'updates':updates,'metrics':score,'partial_epoch':cursor<len(loader)})
        partial=cursor<len(loader)
        if not partial:epoch+=1;cursor=0
        complete=stopped or epoch>=epochs or bad>=cfg.get('patience',2)
        if complete and not updates:raise FloatingPointError('No successful optimizer updates. Inspect FP16 skipped-update logs before generating submissions.')
        save(improved,complete)
        if complete:break
    if args.smoke and rank==0:
        print(json.dumps({'smoke_pass':True,'experiment':cfg['name'],'world_size':world,'device':str(device),'fp16':device.type=='cuda','successful_updates':updates}),flush=True)
    if world>1:dist.destroy_process_group()
if __name__=='__main__':main()
'''
(CODE / 'rr_train.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','rr_train.py')

In [ ]:
MODULE_SOURCE = r'''
"""Direct Phase-2 input restoration, official data, seven RR experiments and exports."""
import gc,json,math,os,shutil,socket,subprocess,sys,time,zipfile
from pathlib import Path
import torch
from legal_models import LegalLM,ModelConfig
from rr_data import DATA_FILES,atomic_json,atomic_torch,digest,file_hash,read_json,submission_zip

SOURCES={'decoder':'01_decoder_full','encoder_decoder':'02_encoder_decoder_full','local_gqa':'03_local_gqa_lora','hybrid':'04_hybrid_partial'}
def merge_phase2(saved):
    cfg=saved['config'];kind=cfg['method'];state={k[len('backbone.'):]:v.float() for k,v in saved['model'].items() if k.startswith('backbone.')}
    if not state:raise ValueError('No backbone in Phase-2 checkpoint.')
    if kind=='adapter':raise ValueError('Nonlinear Phase-2 adapters cannot be folded away. Select a full/partial/LoRA/LoRMA checkpoint.')
    if kind in ('lora','lorma'):
        dense={k:v for k,v in state.items() if not k.endswith(('.base.weight','.A.weight','.B.weight'))}
        for key,value in state.items():
            if not key.endswith('.base.weight'):continue
            name=key[:-len('.base.weight')];A=state[name+'.A.weight'];B=state[name+'.B.weight']
            scale=cfg.get('alpha',16)/cfg.get('rank',8)
            delta=B@(A@value) if kind=='lorma' else B@A
            dense[name+'.weight']=value+scale*delta
        state=dense
    elif kind not in ('full','partial'):raise ValueError('Use a full-size Phase-2 parent, not its compressed KD student.')
    return state
def restore_phase2(root,output,allow_selection=True,overrides=None,allow_tiny=False):
    root=Path(root);output=Path(output);overrides=overrides or {}
    if set(overrides)-set(SOURCES):raise ValueError('Unknown architecture in PARENT_OVERRIDES.')
    if not root.is_dir():raise FileNotFoundError(f'Attach the extracted capstone_phase2 directory: {root}')
    if 'all_kaggle_outputs' in root.parts:raise ValueError('Set PHASE2_ROOT to the direct capstone_phase2 folder.')
    manifest=read_json(root/'phase1/foundation_manifest.json');tokenizer=root/'phase1/tokenizer.json'
    if file_hash(tokenizer)!=manifest['tokenizer_sha256']:raise ValueError('Phase-2 tokenizer/lineage manifest mismatch.')
    output.mkdir(parents=True,exist_ok=True);copied=output/'tokenizer.json';shutil.copy2(tokenizer,copied)
    parents={};records={}
    for arch,run in SOURCES.items():
        refit=root/'runs'/run/'refit/best.pt';selection=root/'runs'/run/'selection/best.pt'
        source=Path(overrides[arch]) if arch in overrides else refit if refit.is_file() else selection if allow_selection else refit
        if not source.resolve().is_relative_to(root.resolve()):raise ValueError('Every Phase-2 checkpoint must be inside PHASE2_ROOT.')
        if not source.is_file():raise FileNotFoundError(f'Missing Phase-2 checkpoint: {source}')
        source_hash=file_hash(source);target=output/f'{arch}.pt'
        if target.exists():
            previous=torch.load(target,map_location='cpu',weights_only=False)
            if previous['provenance']['phase2_sha256']!=source_hash:raise ValueError('Phase-2 source changed; choose a new PROJECT directory.')
            del previous
        else:
            saved=torch.load(source,map_location='cpu',weights_only=False)
            if saved.get('training_origin')!='own_phase1_adapted' or saved['config']['architecture']!=arch:raise ValueError('Expected your own matching Phase-2 NER checkpoint.')
            provenance=saved['provenance'];config=provenance['phase1_model_config']
            if provenance['phase1_sha256']!=manifest['sha256'][arch] or config['architecture']!=arch:raise ValueError('Phase-1/2 lineage does not match.')
            if saved['config']['method']=='kd':raise ValueError('Select a full-size parent checkpoint.')
            state=merge_phase2(saved);model=LegalLM(ModelConfig(**config));model.load_state_dict(state,strict=True)
            count=sum(p.numel() for p in model.parameters())
            if not allow_tiny and not 50_000_000<=count<=300_000_000:raise ValueError('Parent violates the foundation parameter bounds.')
            atomic_torch(target,{'config':config,'model':state,'provenance':{'architecture':arch,'phase2_checkpoint':str(source),
                'phase2_sha256':source_hash,'phase1_sha256':provenance['phase1_sha256'],'tokenizer_sha256':manifest['tokenizer_sha256'],
                'source_stage':saved['stage'],'source_parameters':count}})
            del model,state,saved;gc.collect()
        parents[arch]=str(target);records[arch]={'source':str(source),'sha256':source_hash,'used_selection_fallback':source==selection}
    atomic_json(output/'input_manifest.json',{'input_root':str(root),'input_mode':'direct_directory_only','tokenizer_sha256':file_hash(copied),'parents':records})
    return parents,str(copied)

def obtain_data(project,local_root=None,revision=None):
    destination=Path(project)/'data';destination.mkdir(parents=True,exist_ok=True)
    provenance_path=destination/'source.json'
    if local_root:
        source=Path(local_root)
        if (source/'Phase_3_RR').is_dir():source/='Phase_3_RR'
        missing=[name for name in DATA_FILES if not (source/name).is_file()]
        if missing:raise FileNotFoundError('Missing official Phase-3 files: '+', '.join(missing))
        for name in DATA_FILES:
            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.resolve()!=(source/name).resolve():shutil.copy2(source/name,target)
        origin={'kind':'attached_official_phase3','source':str(source)}
    else:
        from huggingface_hub import HfApi,hf_hub_download
        token=os.environ.get('HF_TOKEN')
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:token=None
        repo='Exploration-Lab/CS-781-Capstone'
        old=read_json(provenance_path) if provenance_path.exists() else None
        commit=old['revision'] if old and old.get('kind')=='official_huggingface' else revision
        try:
            commit=HfApi(token=token).dataset_info(repo,revision=commit).sha
            origin={'kind':'official_huggingface','repo':repo,'revision':commit};atomic_json(provenance_path,origin)
            for name in DATA_FILES:
                target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists():
                    source=hf_hub_download(repo,'Phase_3_RR/'+name,repo_type='dataset',revision=commit,token=token)
                    shutil.copy2(source,target)
        except Exception as exc:raise RuntimeError('Attach the seven official Phase-3 files, or accept the official dataset terms and add an HF_TOKEN Kaggle secret. Do not put tokens in notebook code.') from exc
    origin['sha256']={name:file_hash(destination/name) for name in DATA_FILES};atomic_json(provenance_path,origin)
    return str(destination)
def experiments(parents,settings):
    common={'label_count':13,'head_dim':settings['head_dim'],'dropout':.25,'encoder_dropout':.05,
        'chunk_batch':settings['chunk_batch'],'encoder_checkpointing':settings['encoder_checkpointing'],
        'head_batch':8,'batch_blocks':settings['batch_blocks'],'accumulation':settings['accumulation'],
        'encoder_lr':2e-5,'head_lr':2e-4,'weight_decay':.03,'ce_weight':.5,'shift_weight':.15,
        'epochs':settings['adapt_epochs'],'patience':2,'max_stage_hours':settings['adapt_stage_hours'],
        'rank':8,'alpha':16.,'seed':230437,'freeze_backbone':False,'checkpoint_every':100,'log_every':20,
        'token_bidirectional':settings.get('token_bidirectional',False)}
    specs=[('01_decoder_full','decoder','full','bilstm'),('02_encoder_decoder_full','encoder_decoder','full','transformer'),
        ('03_local_gqa_lora','local_gqa','lora','bilstm'),('04_hybrid_rslora','hybrid','rslora','bilstm'),
        ('05_decoder_lorma','decoder','lorma','bilstm'),('06_encoder_decoder_ia3','encoder_decoder','ia3','bilstm'),
        ('07_decoder_student_kd','decoder','kd','bilstm')]
    rows=[]
    for name,arch,method,head in specs:
        cfg={**common,'name':name,'architecture':arch,'method':method,'head_type':head,'parent':parents[arch]}
        if method in ('lora','rslora','lorma','ia3'):cfg['encoder_lr']=1e-4
        if method=='kd':cfg.update({'student_layers':settings['student_layers'],'temperature':2.,'kd_alpha':.5})
        rows.append(cfg)
    return rows
def launch(script,args,project,gpus=2,log=None):
    with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
    cmd=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1',f'--master_port={port}',
         f'--nproc_per_node={gpus}',str(Path(project)/'code'/script),*map(str,args)]
    env=dict(os.environ);env['PYTHONPATH']=str(Path(project)/'code')+os.pathsep+env.get('PYTHONPATH','');env['TOKENIZERS_PARALLELISM']='false'
    handle=Path(log).open('a',encoding='utf-8') if log else None
    try:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in process.stdout:
            print(line,end='',flush=True)
            if handle:handle.write(line);handle.flush()
        if process.wait()!=0:raise RuntimeError(f'{script} failed. Inspect {log or "cell output"}; recovery checkpoints are retained.')
    finally:
        if handle:handle.close()
def warm_config(cfg,settings):
    return {**cfg,'freeze_backbone':True,'head_lr':6e-4,'epochs':settings['head_epochs'],'patience':4,
        'max_stage_hours':settings['head_stage_hours'],'batch_blocks':16,'accumulation':1}
def run_all(project,rows,cache_path,settings):
    from rr_predict import select_calibration,decode,domain_metrics
    project=Path(project);cache=torch.load(cache_path,map_location='cpu',weights_only=False);catalog=[];gpus=settings['gpus']
    for original in rows:
        cfg=dict(original);root=project/'runs'/cfg['name'];root.mkdir(parents=True,exist_ok=True)
        teacher_train=teacher_all=None
        if cfg['method']=='kd':
            teacher=max(catalog,key=lambda item:item['dev_metrics']['macro_f1']);cfg['teacher_parameters']=teacher['parameters']['total']
            for split,checkpoint in [('train',teacher['selection_checkpoint']),('all',teacher['test_checkpoint'])]:
                if split=='all' and not settings['refit_all']:continue
                destination=root/('teacher_'+split)
                launch('rr_predict.py',['--action','logits','--checkpoint',checkpoint,'--cache',cache_path,
                    '--split',split,'--output',destination],project,gpus,root/f'teacher_{split}.log')
                if split=='train':teacher_train=destination/'complete.pt'
                else:teacher_all=destination/'complete.pt'
        feature_key=cfg['architecture']+('_student_'+str(cfg['student_layers']) if cfg['method']=='kd' else '')
        feature_root=project/'features'/feature_key;feature_root.mkdir(parents=True,exist_ok=True)
        feature_cfg={**cfg,'freeze_backbone':True};feature_config=feature_root/'config.json';atomic_json(feature_config,feature_cfg)
        launch('rr_predict.py',['--action','features','--config',feature_config,'--cache',cache_path,'--split','every',
            '--output',feature_root],project,gpus,feature_root/'features.log')
        features=feature_root/'complete.pt';warm=warm_config(cfg,settings);warm_path=root/'warm_config.json';atomic_json(warm_path,warm)
        args=['--config',warm_path,'--cache',cache_path,'--output',root/'warm_selection','--split','train','--features',features]
        if teacher_train:args+=['--teacher',teacher_train]
        launch('rr_train.py',args,project,settings.get('head_gpus',1),root/'warm_selection.log')
        cfg['head_init']=str(root/'warm_selection/best.pt');config=root/'config.json';atomic_json(config,cfg)
        args=['--config',config,'--cache',cache_path,'--output',root/'selection','--split','train']
        if teacher_train:args+=['--teacher',teacher_train]
        launch('rr_train.py',args,project,gpus,root/'selection.log')
        selection=root/'selection/best.pt';summary=read_json(root/'selection/summary.json')
        launch('rr_predict.py',['--action','logits','--checkpoint',selection,'--cache',cache_path,'--split','dev',
            '--output',root/'selected_dev'],project,gpus,root/'selected_dev.log')
        raw=torch.load(root/'selected_dev/complete.pt',map_location='cpu',weights_only=False)
        metrics,calibration,pred=select_calibration(raw['values'],raw['crf'],cache)
        atomic_json(root/'dev_metrics.json',metrics);atomic_json(root/'domain_metrics.json',domain_metrics(pred,cache));atomic_json(root/'calibration.json',calibration)
        atomic_json(root/'dev_prediction.json',pred);test_checkpoint=selection
        if settings['refit_all']:
            best_warm=max(1,read_json(root/'warm_selection/summary.json')['best_epoch'])
            args=['--config',warm_path,'--cache',cache_path,'--output',root/'warm_refit','--split','all','--epochs',best_warm,'--features',features]
            if teacher_all:args+=['--teacher',teacher_all]
            launch('rr_train.py',args,project,settings.get('head_gpus',1),root/'warm_refit.log')
            refit_cfg={**cfg,'head_init':str(root/'warm_refit/best.pt')};refit_config=root/'refit_config.json';atomic_json(refit_config,refit_cfg)
            args=['--config',refit_config,'--cache',cache_path,'--output',root/'refit','--split','all','--epochs',max(1,summary['best_epoch'])]
            if teacher_all:args+=['--teacher',teacher_all]
            launch('rr_train.py',args,project,gpus,root/'refit.log');test_checkpoint=root/'refit/best.pt'
        launch('rr_predict.py',['--action','logits','--checkpoint',test_checkpoint,'--cache',cache_path,'--split','test',
            '--output',root/'test_prediction'],project,gpus,root/'test_prediction.log')
        test_raw=torch.load(root/'test_prediction/complete.pt',map_location='cpu',weights_only=False)
        prediction=decode(test_raw['values'],test_raw['crf'],cache['labels'],calibration)
        zip_path=project/'submissions'/f"{cfg['name']}.zip";sha=submission_zip(prediction,{did:cache['docs'][did] for did in cache['splits']['test']},cache['labels'],zip_path)
        atomic_json(root/'prediction.json',prediction)
        item={'name':cfg['name'],'architecture':cfg['architecture'],'method':cfg['method'],'dev_metrics':metrics,
            'parameters':summary['parameters'],'selection_epoch':summary['best_epoch'],'selection_checkpoint':str(selection),
            'test_checkpoint':str(test_checkpoint),'submission':str(zip_path),'prediction_sha256':sha,'dev_used_for_calibration':True}
        catalog.append(item);atomic_json(project/'catalog.json',catalog)
        print('READY TO DOWNLOAD:',zip_path,'development-tuned Macro-F1:',round(metrics['macro_f1'],5),flush=True)
    if len(catalog)!=7 or len({item['architecture'] for item in catalog})!=4:raise ValueError('Experiment coverage is incomplete.')
    if len({item['prediction_sha256'] for item in catalog})<7:
        print('Some predictions are identical. Renamed ZIPs do not count as distinct submissions; inspect metrics before uploading.',flush=True)
    return catalog
def smoke_all(project,rows,cache_path,settings):
    for cfg in rows:
        root=Path(project)/'smoke'/cfg['name'];root.mkdir(parents=True,exist_ok=True);config=root/'config.json';atomic_json(config,cfg)
        stamp=digest([cfg,file_hash(cache_path),settings['gpus'],file_hash(Path(project)/'code/rr_models.py'),file_hash(Path(project)/'code/rr_train.py')])
        marker=root/'passed.json'
        if marker.exists() and read_json(marker)['identity']==stamp:continue
        launch('rr_train.py',['--config',config,'--cache',cache_path,'--output',root/'run','--smoke'],project,settings['gpus'],root/'smoke.log')
        atomic_json(marker,{'identity':stamp,'gpus':settings['gpus'],'passed':True})
        for name in ('latest.pt','best.pt'):(root/'run'/name).unlink(missing_ok=True)

def benchmark(rows,cache,project,settings):
    from rr_data import Blocks,collate,to_device
    from rr_models import build_model,loss_function,parameter_report
    from rr_predict import autocast
    if not torch.cuda.is_available():raise RuntimeError('Run the throughput preflight on Kaggle GPUs.')
    device=torch.device('cuda',0);records=[];dataset=Blocks(cache,'train')
    longest=max(range(len(dataset)),key=lambda i:sum(len(part) for sentence in dataset[i]['chunks'] for part in sentence))
    for cfg in rows:
        while True:
            model=optimizer=result=loss=batch=scaler=None
            try:
                torch.cuda.empty_cache();model,_=build_model(cfg);model.to(device).train()
                batch=to_device(collate([dataset[longest] for _ in range(cfg['batch_blocks'])]),device)
                if cfg['method']=='kd':batch['teacher']=torch.zeros(*batch['mask'].shape,len(cache['labels']),device=device)
                optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,fused=True)
                scaler=torch.amp.GradScaler('cuda');weights=torch.ones(len(cache['labels']),device=device)
                torch.cuda.reset_peak_memory_stats(device);times=[];success=0
                for attempt in range(12):
                    torch.cuda.synchronize(device);start=time.monotonic();optimizer.zero_grad(set_to_none=True)
                    with autocast(device):result=model(batch);loss=loss_function(model,result,batch,weights,cfg)
                    if not bool(torch.isfinite(loss)):raise FloatingPointError('Preflight loss is not finite.')
                    scaler.scale(loss).backward();scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                    old=scaler.get_scale();scaler.step(optimizer);scaler.update();torch.cuda.synchronize(device)
                    if scaler.get_scale()>=old:
                        success+=1
                        if success>1:times.append(time.monotonic()-start)
                    if success>=4:break
                if success<4:raise FloatingPointError('Could not obtain finite FP16 optimizer steps.')
                peak=torch.cuda.max_memory_reserved(device)/1024**3
                if peak>12.5:raise torch.cuda.OutOfMemoryError('Reserve space for distributed communication.')
                seconds=sum(times)/len(times);per_epoch=math.ceil(len(dataset)/(cfg['batch_blocks']*settings['gpus']))
                estimate=seconds*per_epoch*cfg['epochs']*(2 if settings['refit_all'] else 1)/3600
                record={'name':cfg['name'],'single_gpu_full_block_step_seconds':seconds,'peak_reserved_gib':peak,
                    'rough_adapt_and_refit_hours_excluding_warmup_validation_cache_io_ddp':estimate,'parameters':parameter_report(model),'chunk_batch':cfg['chunk_batch']}
                records.append(record);print(record,flush=True);break
            except torch.cuda.OutOfMemoryError:
                if cfg['chunk_batch']==1:raise RuntimeError('Preflight still exceeds T4 memory. Use a fresh PROJECT with fewer block_sentences.')
                cfg['chunk_batch']=max(1,cfg['chunk_batch']//2)
            finally:
                del model,optimizer,result,loss,batch,scaler;gc.collect();torch.cuda.empty_cache()
    atomic_json(Path(project)/'benchmark.json',records)
    return records
def bundle_working(working,output):
    working=Path(working);output=Path(output);tmp=output.with_suffix('.tmp.zip')
    files=[p for p in working.rglob('*') if p.is_file() and p.resolve() not in (output.resolve(),tmp.resolve()) and '__pycache__' not in p.parts]
    if shutil.disk_usage(working).free<sum(p.stat().st_size for p in files)+128*1024**2:raise RuntimeError('Download the seven small submission ZIPs first; there is insufficient disk space for a full working archive.')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
        for p in files:z.write(p,p.relative_to(working))
    tmp.replace(output)
    with zipfile.ZipFile(output) as z:
        if z.testzip():raise ValueError('Working archive CRC failure.')
    return str(output)
'''
(CODE / 'rr_workflow.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','rr_workflow.py')

In [ ]:
MODULE_SOURCE = r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.

Usage:
  python evaluate_phase3.py \
      --truth-cl cl_test_truth.json \
      --truth-it it_test_truth.json \
      --prediction prediction.json \
      --labels labels.json

Each truth file must be a JSON object mapping document IDs to ordered lists of
sentence labels. prediction.json must have the same structure and contain
predictions for documents from both reference files.

labels.json may be either:
  ["Label1", "Label2", ...]
or a label->id / id->label mapping.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            # label -> integer ID
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            # integer/string ID -> label
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument(
        "--truth-cl",
        required=True,
        help="Local held-out reference JSON for the CL split.",
    )
    parser.add_argument(
        "--truth-it",
        required=True,
        help="Local held-out reference JSON for the IT split.",
    )
    parser.add_argument(
        "--prediction",
        required=True,
        help="Participant prediction JSON covering documents from both splits.",
    )
    parser.add_argument(
        "--labels",
        required=True,
        help="Public Phase 3 label vocabulary JSON.",
    )
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [
            f"missing={len(missing)}",
            f"extra={len(extra)}",
        ]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''
(CODE / 'evaluate_phase3_official.py').write_text(MODULE_SOURCE.lstrip('\n'),encoding='utf-8')
print('Wrote','evaluate_phase3_official.py')

In [ ]:
importlib.invalidate_caches()
for name in ('rr_workflow','rr_train','rr_predict','rr_models','rr_data','legal_models'): sys.modules.pop(name,None)
from rr_workflow import restore_phase2, obtain_data, experiments, benchmark, smoke_all, run_all, bundle_working
from rr_data import prepare_cache, atomic_json, read_json, digest
parents,tokenizer_path=restore_phase2(PHASE2_ROOT,PROJECT/'parents',ALLOW_SELECTION_FALLBACK,PARENT_OVERRIDES)
manifest=read_json(PROJECT/'parents/input_manifest.json')
print('Selected Phase-2 sources:',json.dumps(manifest['parents'],indent=2))
print('Unchanged original tokenizer:',tokenizer_path)

## Official data and fixed development splits

Both domains use their supplied training and development files. Training excludes exact whole-document duplicates of development examples. Public test labels are never read. Token chunks preserve full sentences and blank sentences retain an output position. The primary metric pools CL+IT sentences and averages over all **13** labels, including zero-support labels. Per-domain metrics are supplementary.

The loader accepts document records, a single document object, ID-keyed records, column-based exports and sentence-record payloads. It validates IDs and label alignment without resegmenting text. For a session using the original notebook, run `Phase3_Data_Loader_Fix_Cell.py` immediately before this data cell; this updated notebook embeds that fix already.

In [ ]:
data_root=obtain_data(PROJECT,DATA_ROOT,DATA_REVISION)
CACHE_PATH=PROJECT/'rr_cache_v1.pt'
cache=prepare_cache(data_root,tokenizer_path,CACHE_PATH,SETTINGS['chunk_tokens'],SETTINGS['block_sentences'],SETTINGS['block_overlap'])
print('Data audit:',json.dumps(cache['audit'],indent=2))
print('Document counts:',{split:len(ids) for split,ids in cache['splits'].items()})
print('Training/development label support:',json.dumps(cache['support'],indent=2))
ceiling=sum(value>0 for value in cache['support']['dev'].values())/len(cache['labels'])
print('Maximum possible development Macro-F1 with this support:',round(ceiling,4))
print('Training-absent roles:',[label for label,n in cache['support']['train'].items() if n==0])
print('Document-block counts:',{split:len(cache[split+'_blocks']) for split in ('train','dev','all','test')})
atomic_json(PROJECT/'annotation_audit.json',cache['audit'])
atomic_json(PROJECT/'run_settings.json',SETTINGS)
atomic_json(PROJECT/'labels.json',cache['labels'])
for domain in ('CL','IT'):
    truth={did:cache['docs'][did]['gold'] for did in cache['splits']['dev'] if cache['docs'][did]['domain']==domain}
    atomic_json(PROJECT/f'dev_{domain}_truth.json',truth)

## Seven methods and actual GPU preflight

All four parent families participate. Methods are full fine-tuning, LoRA, rsLoRA, compulsory LoRMA+, IA³ and smaller-student knowledge distillation. Candidate 02 tests a sentence Transformer; the other six use a BiLSTM context head. All have learned CRF decoding and auxiliary role-change supervision. Cached warmups train only the new head; subsequent stages update the declared backbone parameters.

The benchmark allocates optimizer state and measures real FP16 updates on GPU 0, reducing token-chunk batch size if needed. It estimates adaptation/refit compute only. The two-process smoke test then checks every method. Zero teacher logits exercise only the KD preflight; real training requires the selected larger RR teacher’s split-matched cache.

In [ ]:
rows=experiments(parents,SETTINGS)
for row in rows: print(row['name'],row['architecture'],row['method'],row['head_type'])
if RUN_BENCHMARK:
    timing=benchmark(rows,cache,PROJECT,SETTINGS)
    print('Rough adaptation + refit hours, excluding caches/warmup/validation/inference/DDP/IO:',
          round(sum(item['rough_adapt_and_refit_hours_excluding_warmup_validation_cache_io_ddp'] for item in timing),2))
if RUN_GPU_SMOKE: smoke_all(PROJECT,rows,CACHE_PATH,SETTINGS)
atomic_json(PROJECT/'experiment_plan.json',rows)

## Train, select and save each candidate immediately

Frozen features are cached per parent. Head warmups use one GPU and larger block batches; adaptation and prediction use both T4s. A development-only grid selects CRF transition strength and a small training-prior correction. Optional train+dev refits use the selected epoch counts to produce test outputs. The displayed development score remains the earlier held-out result and is explicitly development-tuned, not an independent test estimate.

Run this cell once; rerunning the same configuration resumes saved work. A failure in a later candidate leaves earlier ZIPs intact. The student distills RR emissions from the best larger RR candidate, with separate training-only and all-label teacher caches.

In [ ]:
catalog=run_all(PROJECT,rows,CACHE_PATH,SETTINGS)
print('Finished seven experiments.')
for item in catalog:
    print(item['name'],'development-tuned Macro-F1',round(item['dev_metrics']['macro_f1'],5),item['submission'])
    display(FileLink(item['submission']))

## Official development evaluation and comparisons

This invokes the supplied official script on the public CL/IT development truth and each saved prediction. The output is numeric verification, not a generated interpretation of your results. Review per-label/domain tables and write your own analysis under the course rules.

In [ ]:
catalog=read_json(PROJECT/'catalog.json')
for item in catalog:
    print(item['name'])
    subprocess.run([sys.executable,str(CODE/'evaluate_phase3_official.py'),
        '--truth-cl',str(PROJECT/'dev_CL_truth.json'),'--truth-it',str(PROJECT/'dev_IT_truth.json'),
        '--prediction',str(PROJECT/'runs'/item['name']/'dev_prediction.json'),
        '--labels',str(PROJECT/'labels.json')],check=True)
columns=['name','architecture','method','dev_macro_f1','dev_macro_precision','dev_macro_recall','parameters','trainable_parameters','prediction_sha256']
with (PROJECT/'comparison.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=columns);writer.writeheader()
    for item in catalog:
        writer.writerow({'name':item['name'],'architecture':item['architecture'],'method':item['method'],
            'dev_macro_f1':item['dev_metrics']['macro_f1'],'dev_macro_precision':item['dev_metrics']['macro_precision'],
            'dev_macro_recall':item['dev_metrics']['macro_recall'],'parameters':item['parameters']['total'],
            'trainable_parameters':item['parameters']['trainable'],'prediction_sha256':item['prediction_sha256']})
distinct=len({item['prediction_sha256'] for item in catalog})
print('Prediction sets:',len(catalog),'Distinct prediction sets:',distinct)
if distinct<7: print('Inspect and improve duplicate candidates: identical predictions do not count as separate submissions.')
display(FileLink(str(PROJECT/'comparison.csv')))

## Download the seven ZIPs and preserve the complete working directory

Upload each `submissions/<candidate>.zip` individually to Codabench. The download collection below contains the seven ZIPs and is **not** itself a platform submission. The full working archive retains dense parents needed by compact adapter checkpoints, code, metadata, features and recovery states for later phases. Restore it as files, then attach the extracted `capstone_phase3` folder for a future notebook.

In [ ]:
catalog=read_json(PROJECT/'catalog.json')
if len(catalog)!=7: raise RuntimeError('Complete all seven candidates before exporting the collection.')
collection=WORKING/'phase3_seven_submissions.zip'
with zipfile.ZipFile(collection,'w',zipfile.ZIP_DEFLATED) as archive:
    for item in catalog: archive.write(item['submission'],Path(item['submission']).name)
display(FileLink(str(collection)))
if MAKE_FULL_WORKING_ARCHIVE:
    complete=bundle_working(WORKING,WORKING/'phase3_all_working.zip')
    display(FileLink(complete))

# Phase 3: seven rhetorical-role submissions on Kaggle T4 ×2

Prepared from the current [CS781 capstone brief](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), last modified October 13, 2026, and its [official Phase-3 evaluator](https://drive.google.com/file/d/1d634sL8o-AmnCa7Iy5umIMMJ3KifFOHY/view). The brief gives **October 16, 2026, 11:59 PM IST** as the deadline. It says evaluation may take 15–30 minutes or longer. Generating ZIPs does not upload them to Codabench.

The updated Important Instructions require six or more submissions with genuinely different strategies, while the Phase-3 section still mentions four. This notebook runs seven experiments, covers all four foundation families, and includes compulsory LoRMA and a smaller distilled student. Identical predictions do not count as separate submissions; the notebook detects duplicates and reports them without inventing label changes.

**Course-use note:** the brief restricts LLM use for core architecture code, tuning and result analysis, and requires an LLM declaration. This notebook contains AI-written architecture and training code. Do not represent it as independently authored code; check its permitted use with the teaching team and follow the declaration requirement. The research summaries explain design choices, rather than supplying conclusions about your unobserved results.

## Inputs: your saved Phase-2 output dataset

Default root:

```text
/kaggle/input/datasets/hardik4tiwari/phase-2-outputs/capstone_phase2/
    phase1/
        tokenizer.json
        foundation_manifest.json
    runs/
        01_decoder_full/refit/best.pt
        02_encoder_decoder_full/refit/best.pt
        03_local_gqa_lora/refit/best.pt
        04_hybrid_partial/refit/best.pt
```

Other files in the saved output can remain. The four `selection/best.pt` files can substitute when refits were not run, controlled by `ALLOW_SELECTION_FALLBACK`. Selected paths are printed. An optional `PARENT_OVERRIDES` dictionary can point to another compatible full-sized Phase-2 checkpoint **inside the same root**, for example the optional decoder span model. No recursive checkpoint discovery, input ZIP extraction or `all_kaggle_outputs` input is used. If Kaggle mounts the dataset at `/kaggle/input/phase-2-outputs/`, edit **only `PHASE2_ROOT`**.

You do not need the original Phase-1 dataset attached separately when these files are present. The Phase-2 checkpoints contain the backbone weights and original model configuration. The copied manifest supplies tokenizer and Phase-1 lineage hashes. The notebook merges Phase-2 additive LoRA or multiplicative LoRMA updates into dense backbone weights before adding a new RR adaptation. It does not silently discard learned low-rank changes. Nonlinear Phase-2 adapters cannot be removed this way and are rejected as parent overrides. The four default checkpoints are compatible.

The Phase-3 official data consists of seven files:

```text
Phase_3_RR/
    Train/
        CL_train.json
        CL_dev.json
        IT_train.json
        IT_dev.json
        label_vocab.json
    Test/
        CL_test.json
        IT_test.json
```

Set `DATA_ROOT` to an attached folder with this structure, or leave it `None`, enable Internet, accept the terms for `Exploration-Lab/CS-781-Capstone`, and add an `HF_TOKEN` Kaggle secret. The notebook resolves and records a dataset commit and all seven file hashes. It downloads data and small Python dependencies, never external model weights, tokenizers or LLM predictions. Real gated examples and your actual trained Phase-2 weights were not available in the development environment; schema, lineage and sentence audits run on Kaggle.

### Loader correction for existing sessions

The updated data module accepts a document-record list, one document object, an ID-to-document mapping, Hugging Face-style `id`/`text`/`labels` columns, and ID-keyed text/label columns (including pandas row-index columns with an explicit ID column). An ID-keyed document may also contain sentence records, `[sentence, label]` pairs, or an unambiguous `[sentences, labels]` payload. Original string IDs, sentence order, blank sentences and label vocabulary are preserved. Missing IDs, mismatched columns, partial annotations, conflicting fields and ambiguous payloads fail with a structural diagnostic. Raw document strings are never silently split into sentences.

If the original notebook fails with `Document must contain id/text/labels fields.`, copy the complete contents of `Phase3_Data_Loader_Fix_Cell.py` into a new cell immediately before the failed data-preparation cell. Run the fix, then rerun data preparation and continue. The fix updates only the working `code/rr_data.py` parser, reloads the module and refreshes the notebook's `prepare_cache` reference; no kernel restart or deletion of checkpoints is required. Do not rerun the old embedded `rr_data.py` writing cell afterward because it would overwrite the fix. The updated full notebook already contains it. This traceback does not reveal the actual JSON layout; structural regression tests cover supported layouts, while the real-data audit must still run on Kaggle.

## Seven experiments

| ZIP stem | Parent family | Adaptation | Sentence-context head |
|---|---|---|---|
| 01_decoder_full | Phase-2 decoder | Full fine-tuning | BiLSTM + learned CRF |
| 02_encoder_decoder_full | Phase-2 encoder–decoder | Full fine-tuning | Small sentence Transformer + CRF |
| 03_local_gqa_lora | Phase-2 local GQA | Additive LoRA | BiLSTM + CRF |
| 04_hybrid_rslora | Phase-2 LSTM–Transformer | Rank-stabilized LoRA | BiLSTM + CRF |
| 05_decoder_lorma | Phase-2 decoder | LoRMA+ multiplicative adaptation | BiLSTM + CRF |
| 06_encoder_decoder_ia3 | Phase-2 encoder–decoder | IA³ scaling vectors | BiLSTM + CRF |
| 07_decoder_student_kd | Compressed Phase-2 decoder | Hard labels + RR teacher-logit distillation | BiLSTM + CRF |

These are six adaptation strategies across seven experiments, not seven learning-rate settings. LoRA, rsLoRA, LoRMA+, IA³ and KD are distinct approaches listed by the brief. LoRMA+ is named precisely rather than claiming the permutation variant. The student keeps four uniformly spaced blocks from the eight-block decoder, copying embeddings and final normalization. Actual total and trainable counts are recorded. Its teacher is the larger RR model with the best development-tuned Macro-F1 among the first six candidates. Phase-2 NER logits are not reused as RR targets because their label space is different.

## Architecture and accuracy choices

Each supplied sentence is tokenized with your unchanged tokenizer. Long sentences are divided into 128-token chunks, retaining **all** tokens. Empty sentences receive an unknown-token representation and retain their position in the output. Each chunk is encoded by your own legal backbone, preserving its causal/local/cross-attention pattern by default. Chunk mean, maximum and last-token features are concatenated; chunk summaries are averaged with token-count weights to form one sentence vector. This preserves text coverage without treating an entire long legal document as one quadratic token-attention matrix. The new sentence-context layer is bidirectional. Optional `token_bidirectional=True` changes token attention too; use a new PROJECT and measure that architectural change rather than assuming it helps.

A new document-context head combines sentence vectors with the CL/IT domain embedding and continuous document-position features. It processes overlapping 24-sentence blocks, with eight sentences of overlap by default. Six candidates use a two-layer BiLSTM; candidate 02 uses a two-layer sentence Transformer. This is bounded local document context, not unrestricted token attention over the entire document. Inference merges overlapping sentence emissions and performs **one learned CRF decode across the complete document**. Every original sentence receives exactly one label, in its original order.

The supervised loss combines token-normalized CRF negative log likelihood, moderately class-weighted sentence cross-entropy, and a binary adjacent-label-change objective. Class weights use public training support only. The change objective uses the two contextual states and their absolute difference. It is a shared-representation auxiliary-task variant, not an exact reproduction of papers with separate label-shift encoders. No gold role or gold shift is provided as an inference input.

The supplied CL/IT development files remain held out during selection. Exact duplicated whole documents are removed from training if present in development. Selection uses pooled sentence-level Macro-F1 over **all 13 labels**, matching the evaluator. It is not an average of the two domain Macro-F1 values. CL and IT metrics, per-label support, confusion matrices and the maximum attainable development Macro-F1 are reported separately. A zero-support label remains in the denominator.

Checkpoint selection uses raw CRF decoding. A small nine-choice development grid then chooses CRF transition strength and a training-prior-based label bias. This explicitly reuses development labels for selection/calibration; its displayed score is not an independent test estimate. Optional refitting trains on public train+dev labels, with selected epoch counts, solely to generate test predictions. Refit weights never supply the claimed development score. No target score, including 0.75–0.80, is guaranteed by the implementation or cited papers.

## Research reviewed and how it changes the implementation

1. **Malik et al., 2022, Semantic Segmentation of Legal Documents via Rhetorical Roles** — [paper](https://aclanthology.org/2022.nllp-1.13.pdf), Sections 4–5. The paper studies CL/IT legal roles and adjacent role-shift information. It motivates the sentence-sequence head and auxiliary change loss. Our shared small shift head differs from its separate shift-representation branches. The reported shift-prediction F1 is not the main RR Macro-F1 and is not a target score for this notebook.

2. **Chen et al., 2023, YNU-HPCC at LegalEval** — [paper](https://aclanthology.org/2023.semeval-1.286.pdf), Sections 2–3. Word/sentence hierarchy and CRF prediction motivate contextual sentence labeling. We replace its externally pretrained LEGAL-BERT and word BiLSTM with your own foundation and inexpensive chunk pooling. Its task setup and backbone differ; its ranking does not predict your score.

3. **LegalSeg, NAACL Findings 2025** — [paper](https://aclanthology.org/2025.findings-naacl.63.pdf), Sections 5–6. Its sentence-Transformer, positional context, hierarchical CRF and multi-task comparisons motivate candidate 02 and the shared context features. Its results also show that extra architectural complexity is not automatically beneficial. Its seven-role setup must not be compared numerically with this thirteen-role competition.

4. **IL-TUR, Joshi et al., 2024** — [paper](https://arxiv.org/html/2407.05399v2), Appendices A.2 and B.2. Domain variation, label imbalance and sentence-sequence modeling motivate pooled scoring plus domain diagnostics. We use the competition label vocabulary and evaluator as the authority, rather than silently inheriting another dataset’s names or scoring convention.

5. **Hu et al., LoRA** — [paper](https://arxiv.org/html/2106.09685v2). Candidate 03 uses `W x + (alpha/r) B A x`, with random A and zero B, training selected attention/feed-forward projections and the new RR head. Frozen base weights reduce optimizer and gradient storage; activations still require work. Our benchmark measures actual T4 steps instead of equating parameter savings with equal speed gains.

6. **Kalajdzievski, rsLoRA** — [paper](https://arxiv.org/html/2312.03732v1). Candidate 04 changes additive-adapter scaling to `alpha/sqrt(r)`, as motivated by rank-stable learning. It is a distinct scaling method rather than a claimed larger model. At a fixed small rank, its quality is empirical and is not assumed superior to LoRA.

7. **Bihany, Patel and Modi, LoRMA, ACL Findings 2025** — [paper](https://aclanthology.org/2025.findings-acl.527.pdf), method and complexity sections. Candidate 05 implements LoRMA+ as `y + (alpha/r) B(Ay)` for `y=W x`. It applies factors on the output side and never forms a large square multiplier during training. Merging a Phase-2 LoRMA parent evaluates `B(AW)` in that order. The paper’s convergence observations are not promised wall-clock speedups on RR/T4.

8. **Liu et al., IA³ / T-Few, 2022** — [paper](https://arxiv.org/html/2205.05638v2). Candidate 06 trains vectors on key/value outputs and feed-forward intermediate activations, represented by scaling the input to the down projection. Vectors start at one so the original function is preserved. We use this adaptation mechanism with a discriminative RR head, not the complete few-shot prompt-based T-Few system.

9. **Hinton, Vinyals and Dean, 2015** — [paper](https://arxiv.org/abs/1503.02531). Candidate 07 combines hard-label supervision with softened RR emission-logit KL divergence and the temperature-squared correction. Teacher logits are cached once and identified by checkpoint, data fingerprint and training split. A selection teacher supplies training-only targets; an all-label refit teacher supplies the separate final refit targets.

10. **Kumar et al., LP-FT, ICLR 2022** — [paper](https://arxiv.org/pdf/2202.10054). Its analysis motivates learning a fresh task head before updating pretrained features. Our warmup is a nonlinear hierarchical head, not a literal linear probe, and its vision-domain findings do not establish an RR gain. The practical saving is that frozen-backbone embeddings can be reused across warmup epochs and compatible adaptation routes.

11. **Chen et al., activation checkpointing, 2016** — [paper](https://arxiv.org/abs/1604.06174). Optional non-reentrant checkpointing recomputes token-encoder chunks during backward to bound activation memory. It is a memory/time tradeoff. It does not claim the sequence-parallel or selective-recomputation implementation used by much larger model systems.

12. **Micikevicius et al., mixed precision, 2017** — [paper](https://arxiv.org/abs/1710.03740). GPU token/head operations use FP16 autocast and loss scaling while losses and CRF recurrences use FP32. Successful updates and skipped updates are tracked. Two T4s hold two replicas; they are not one 30 GB memory pool.

13. **Dao et al., FlashAttention, 2022** — [paper](https://arxiv.org/abs/2205.14135). Efficient exact attention motivates using PyTorch’s native scaled-dot-product attention. No FlashAttention-2 extension or particular fused backend is assumed on Turing T4 hardware. Masks, shapes, dtype and dropout determine dispatch; measured timings are authoritative.

14. **BehnamGhader et al., LLM2Vec, 2024** — [paper](https://arxiv.org/abs/2404.05961). The paper finds that simply removing a causal mask can hurt representations and uses additional adaptation stages. This motivates preserving inherited token masks by default while providing bidirectional sentence context. The optional token-mask switch does not implement LLM2Vec’s masked-next-token or contrastive learning recipe.

## Training time, memory and recovery

The notebook first measures full-block training steps and peak reserved memory for each method on GPU 0, with optimizer state allocated. These forecasts exclude feature preparation, warmup, development inference, KD caches, distributed communication and file writing. An actual two-process smoke update then exercises every candidate before training. Fake zero teacher logits are used only for the KD smoke/preflight; the real student run refuses to start without its actual larger RR teacher cache.

Frozen sentence features are computed once per parent (and once for the smaller student), stored in FP16 and reused only while the backbone is frozen. Warmup batches contain 16 sentence blocks and use one GPU to avoid unnecessary synchronization of a small head. Adaptation, feature extraction and inference use both T4s. After adaptation, sentence features are recomputed from the current checkpoint rather than reusing stale parent features. Large caches checkpoint every eight documents, and completed caches are reused. Frozen-backbone/delta checkpoints omit redundant base weights; keep the `parents/` directory with them.

Default warmup: at most 20 epochs, patience 4, a **0.04-hour** loop cap per warmup stage. Adaptation: at most 6 epochs, patience 2, a **0.15-hour** loop cap per adaptation stage. With selection and refit, seven runs have up to `14 × (0.04 + 0.15) = 2.66` hours inside these loops. Validation at a stage boundary can exceed a cap; loading, preprocessing, caches, inference and archive creation add time. This is a budget allowance, not a measured total runtime. Set `refit_all=False` before starting if you need to produce all seven ZIPs sooner; that removes the warm-refit and adaptation-refit stages. Plan around the competition deadline and measured Kaggle throughput.

Each completed candidate immediately saves its ZIP and updates `catalog.json`; a later failure does not erase earlier submissions. Training checkpoints retain optimizer, scaler, sampler cursor and per-rank random states. Cache/config/source identities prevent incompatible resumes. Keep settings and GPU counts fixed when resuming. Changing the design requires a new `PROJECT` directory. After a session reset, restore the complete working archive and reattach the original input dataset. Smoke-test weights are discarded after their successful test; logs and pass records remain.

## Outputs and verification limits

Seven small `submissions/*.zip` files each contain only `prediction.json`. The notebook validates exact document IDs, official labels and sentence counts, runs the official evaluator on public CL/IT development predictions, writes a comparison CSV, and exports a final whole-working-directory archive. Upload the seven individual submission ZIPs to the platform. The complete working archive is for downloads, recovery and later phases, not a competition prediction submission.

Synthetic own-origin Phase-2 checkpoints exercise checkpoint folding, all seven losses/gradients, CRF likelihood/Viterbi, long and empty sentences, development split isolation, warmup, adaptation, teacher caches, refitting, prediction and ZIP structure. These tests establish compatibility and behavior, not real-data quality. The actual T4 smoke test and real-data audits must pass on Kaggle. Inspect the printed per-class and per-domain results; a method should be judged by those measurements rather than its name or a score from a different paper.

Additional checks matched the official evaluator exactly, verified frozen parameters and padding invariance, reproduced uninterrupted model/optimizer tensors after a saved-step interruption, and rejected a teacher cache from the wrong split. Two-process CPU/Gloo initialization was blocked by this runtime's network-interface permissions. Actual two-T4 execution and real-data accuracy therefore remain Kaggle checks; no multi-GPU validation success is claimed from the development environment.